# Qstate lab: keep track of quantum states

This lab teaches the qstate layer through the path you will usually use: `QuantumStateManager`. The manager stores states, owns subsystem names, applies gates and noise, samples measurements, and keeps refs/layouts consistent while the rest of the simulator moves events around it.

In [ ]:
from __future__ import annotations

from random import Random

import numpy as np

from simyuj.qstate import POVM, POVMElement, QuantumStateManager, StateLayout, SubsystemId
from simyuj.qstate.noise import amplitude_damping, depolarizing, phase_flip
from simyuj.qstate.ops import CNOT, H, SWAP, X, Z
from simyuj.qstate.state import bell_fidelity, purity

## 1. Names, refs, and records

A subsystem name is the handle you use in code. A state ref is the store's handle. The record tells you what payload and tensor layout that ref owns.

In [ ]:
def q(name: str) -> SubsystemId:
    return SubsystemId(name)


def short_complex(values):
    return [complex(round(v.real, 3), round(v.imag, 3)) for v in values]


def show_record(manager: QuantumStateManager, state_ref: int, label: str = 'state') -> None:
    record = manager.record(state_ref)
    print(f'{label}: ref={state_ref}, rep={record.rep}')
    print('  subsystems:', tuple(str(s) for s in record.layout.subsystems))
    print('  dims:', record.layout.dims)
    payload = record.payload
    if hasattr(payload, 'vector'):
        print('  ket:', short_complex(payload.vector))
    elif hasattr(payload, 'rho'):
        print('  density diagonal:', np.round(np.diag(payload.rho).real, 3).tolist())
    elif hasattr(payload, 'probs'):
        print('  bell probabilities:', tuple(round(p, 3) for p in payload.probs))

In [ ]:
alice = q('alice-photon')
bob = q('bob-photon')
layout = StateLayout((alice, bob), (2, 2))

print('layout subsystems:', tuple(str(s) for s in layout.subsystems))
print('axis of bob:', layout.axis_of(bob))
print('Hilbert-space dimension:', layout.hilbert_dim)

In [ ]:
manager = QuantumStateManager()
q0 = q('q0')
q1 = q('q1')

ref0 = manager.prepare('|0>', subsystems=(q0,))
ref1 = manager.prepare('|+>', subsystems=(q1,))

print('refs:', ref0, ref1)
print('live records:', manager.size())
show_record(manager, ref0, 'q0')
show_record(manager, ref1, 'q1')
print('q1 location:', manager.location_of(q1))

## 2. Gates change the payload, not the subsystem name

Apply gates to subsystem IDs. If the targets already live in one record, the ref stays the same.

In [ ]:
manager = QuantumStateManager()
q0 = q('work')
state_ref = manager.prepare('|0>', subsystems=(q0,))

manager.apply(H, targets=(q0,))
show_record(manager, state_ref, 'after H')

x_result = manager.measure(targets=(q0,), basis='x')
print('x-basis measurement:', x_result.label, 'probability=', x_result.probability)
show_record(manager, state_ref, 'after x measurement')

In [ ]:
manager = QuantumStateManager()
q0 = q('bit')
state_ref = manager.prepare('|0>', subsystems=(q0,))

manager.apply(X, targets=(q0,))
z_result = manager.measure(targets=(q0,), basis='z')

print('after X, z measurement:', z_result.label)
show_record(manager, state_ref, 'stored state')

## 3. Randomness is explicit

A measurement with two possible outcomes needs an RNG. In timeline code, that RNG should come from `Timeline.rng(...)`; in a notebook, `Random(seed)` is enough to see the rule.

In [ ]:
manager = QuantumStateManager()
q0 = q('coin')
state_ref = manager.prepare('|+>', subsystems=(q0,))

try:
    manager.measure(targets=(q0,), basis='z')
except ValueError as exc:
    print('without rng:', exc)

result = manager.measure(targets=(q0,), basis='z', rng=Random(7))
print('with rng:', result.label, 'probability=', result.probability)
show_record(manager, state_ref, 'collapsed coin')

In [ ]:
manager = QuantumStateManager()
q0 = q('preview')
state_ref = manager.prepare('|+>', subsystems=(q0,))

peek = manager.measure(targets=(q0,), basis='z', rng=Random(7), collapse=False)
print('peeked outcome:', peek.label, 'collapsed?', peek.collapsed)
show_record(manager, state_ref, 'state after collapse=False')

commit = manager.measure(targets=(q0,), basis='z', rng=Random(7), collapse=True)
print('committed outcome:', commit.label, 'post_state_ref=', commit.post_state_ref)
show_record(manager, state_ref, 'state after collapse=True')

## 4. Two-qubit gates can merge separate records

If a gate spans two separate live records, the manager tensors them into one new record and consumes the old refs.

In [ ]:
manager = QuantumStateManager()
control = q('control')
target = q('target')
left_ref = manager.prepare('|1>', subsystems=(control,))
right_ref = manager.prepare('|0>', subsystems=(target,))

merged_ref = manager.apply(CNOT, targets=(control, target))

print('old refs still live?', manager.store.contains_state(left_ref), manager.store.contains_state(right_ref))
print('merged ref:', merged_ref)
show_record(manager, merged_ref, 'after CNOT')
print('z measurement:', manager.measure(targets=(control, target), basis='z').outcome)

In [ ]:
manager = QuantumStateManager()
a = q('a')
b = q('b')
manager.prepare('|0>', subsystems=(a,))
manager.prepare('|1>', subsystems=(b,))

try:
    manager.measure(targets=(a, b), basis='z')
except Exception as exc:
    print('projective measurement before merge:', type(exc).__name__, '-', exc)

merged_ref = manager.apply(SWAP, targets=(a, b))
print('after SWAP merge, z outcome:', manager.measure(targets=(a, b), basis='z').outcome)
show_record(manager, merged_ref, 'merged pair')

## 5. Build and inspect a Bell pair

You can prepare Bell states directly, or build one from gates. The result record still tells you which ref was measured.

In [ ]:
manager = QuantumStateManager()
a = q('alice')
b = q('bob')
bell_ref = manager.prepare('phi+', subsystems=(a, b))

show_record(manager, bell_ref, 'prepared phi+')
print('fidelity with phi+:', round(bell_fidelity(manager.get(bell_ref), 'phi+'), 3))

bell_result = manager.measure_bell(targets=(a, b))
print('Bell measurement:', bell_result.label, bell_result.outcome, 'state_ref=', bell_result.state_ref)

In [ ]:
manager = QuantumStateManager()
a = q('alice')
b = q('bob')
manager.prepare('|0>', subsystems=(a,))
manager.prepare('|0>', subsystems=(b,))

manager.apply(H, targets=(a,))
bell_ref = manager.apply(CNOT, targets=(a, b))

show_record(manager, bell_ref, 'built from H then CNOT')
print('fidelity with phi+:', round(bell_fidelity(manager.get(bell_ref), 'phi+'), 3))

## 6. Rename a subsystem when the physical carrier changes

A photon can become a memory slot without changing the quantum payload. Relabeling updates ownership and layout names.

In [ ]:
manager = QuantumStateManager()
photon = q('photon:A:42')
partner = q('photon:B:42')
memory = q('memory:A:0')

state_ref = manager.prepare('psi-', subsystems=(photon, partner))
old_payload = manager.get(state_ref)
manager.relabel_subsystem(photon, memory)

show_record(manager, state_ref, 'after relabel')
print('same payload object?', manager.get(state_ref) is old_payload)
print('memory location:', manager.location_of(memory))

## 7. Density states and noise

Noise channels act on density states. `apply_noise_models(..., auto_convert=True)` is the friendly route when you start from a ket.

In [ ]:
manager = QuantumStateManager()
q0 = q('relaxing-qubit')
state_ref = manager.prepare('|1>', rep='density', subsystems=(q0,))

print('before noise')
show_record(manager, state_ref)
manager.apply_noise(amplitude_damping(1.0), targets=(q0,))
print('after full amplitude damping')
show_record(manager, state_ref)
print('purity:', round(purity(manager.get(state_ref)), 3))

In [ ]:
manager = QuantumStateManager()
q0 = q('noisy-plus')
state_ref = manager.prepare('|+>', subsystems=(q0,))

manager.apply_noise_models([depolarizing(0.25), phase_flip(0.2)], targets=(q0,))
show_record(manager, state_ref, 'after auto-converted noise')
print('record rep:', manager.record(state_ref).rep)
print('purity:', round(purity(manager.get(state_ref)), 3))

## 8. Reset and discard keep ownership honest

Reset writes a target back into a chosen state. Discard traces out a subsystem and may shrink the record.

In [ ]:
manager = QuantumStateManager()
a = q('left')
b = q('right')
state_ref = manager.prepare('|11>', subsystems=(a, b))

manager.reset(targets=(b,), state='|0>')
print('after reset right to |0>:')
show_record(manager, state_ref)
print('z outcome:', manager.measure(targets=(a, b), basis='z').outcome)

In [ ]:
manager = QuantumStateManager()
a = q('kept')
b = q('discarded')
state_ref = manager.prepare('phi+', subsystems=(a, b))

remaining_ref = manager.discard(targets=(b,))
print('discard returned:', remaining_ref)
show_record(manager, state_ref, 'after discarding one half')
print('store size:', manager.size())

## 9. A small POVM, when projective labels are not enough

POVM measurement writes back density representation when it collapses. This example is just a z-basis detector with custom labels.

In [ ]:
z_detector = POVM(
    (
        POVMElement('dark', [[1, 0], [0, 0]]),
        POVMElement('click', [[0, 0], [0, 1]]),
    ),
    name='toy-z-detector',
)

manager = QuantumStateManager()
q0 = q('detector-input')
state_ref = manager.prepare('|1>', subsystems=(q0,))

result = manager.measure_povm(z_detector, targets=(q0,))
print('POVM label:', result.label, 'probability=', result.probability)
show_record(manager, state_ref, 'after POVM')

## 10. Bell-diagonal states are compact Bell mixtures

Use them when the state is naturally a probability distribution over Bell labels. They are small, easy to inspect, and convert to density when needed.

In [ ]:
manager = QuantumStateManager()
a = q('link-left')
b = q('link-right')
state_ref = manager.prepare(
    {'phi+': 0.72, 'phi-': 0.18, 'psi+': 0.08, 'psi-': 0.02},
    rep='bell_diag',
    subsystems=(a, b),
)

show_record(manager, state_ref, 'Bell mixture')
print('fidelity with phi+:', round(bell_fidelity(manager.get(state_ref), 'phi+'), 3))
print('purity:', round(purity(manager.get(state_ref)), 3))

sample = manager.measure_bell(targets=(a, b), rng=Random(4), collapse=False)
print('sampled Bell label without collapse:', sample.label)
show_record(manager, state_ref, 'mixture still stored')

In [ ]:
manager.convert(state_ref, 'density')
show_record(manager, state_ref, 'Bell mixture as density')
print('density shape:', manager.get(state_ref).rho.shape)

## Mini lab: send three qubits through a tiny state workflow

Try changing `states`, `noise_strength`, or the measurement bases. The prints should tell you which refs merged, which representation is stored, and which outcomes came back.

In [ ]:
states = ['|0>', '|+>', '|1>']
noise_strength = 0.15
bases = ['z', 'x', 'z']

manager = QuantumStateManager()
qubits = tuple(q(f'lab-{i}') for i in range(len(states)))
refs = []

for qubit, state in zip(qubits, states):
    refs.append(manager.prepare(state, subsystems=(qubit,)))

print('prepared refs:', refs)
print('prepared qubits:', [str(qubit) for qubit in qubits])


In [ ]:
manager.apply(H, targets=(qubits[0],))
merged_ref = manager.apply(CNOT, targets=(qubits[0], qubits[1]))
manager.apply_noise_models([depolarizing(noise_strength)], targets=(qubits[2],))

print('after operations:')
print('merged first two qubits into ref:', merged_ref)
for qubit in qubits:
    print(str(qubit), 'owned by ref', manager.state_of(qubit), 'at', manager.location_of(qubit))

In [ ]:
results = []
for qubit, basis in zip(qubits, bases):
    result = manager.measure(targets=(qubit,), basis=basis, rng=Random(10))
    results.append((str(qubit), basis, result.label, round(result.probability, 3)))

print('measurements:', results)
print('final state refs:', sorted({manager.state_of(qubit) for qubit in qubits}))


## Keep this model in your head

Most qstate work is a loop: name subsystems, prepare or receive a ref, apply operations to subsystem IDs, pass explicit RNGs to stochastic measurements, and inspect the record when ownership or representation matters.